# Does a coding agent weaken tests under pressure?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/articles/dark-software-factory-test-authority/gaming_v1.ipynb)

Companion to https://profrod.ai/articles/dark-software-factory-test-authority. There are 30 small functions, each in a solvable version and an impossible one, where one visible test contradicts the docstring. Each runs with tests writable or read-only, and under a neutral or a pressured instruction. Three small open models (Qwen2.5-Coder 1.5B and 3B, Qwen3 1.7B) get up to three attempts each, for 720 recorded runs. Every model-written file ran under an operating-system sandbox.

Part 1 recomputes the article's numbers from the recorded runs, without running any model-written code. Part 2 would rerun a slice; it needs an OS sandbox, which Colab does not provide by default, so it refuses rather than run untrusted code unconfined.

In [ ]:
import os
import urllib.request

BASE = "https://raw.githubusercontent.com/profrodai/profrodai-resources/main/articles/dark-software-factory-test-authority/"
FILES = ["gaming_tasks_v1.py", "gaming_sandbox_v1.py", "gaming_stats_v1.py", "gaming_run_v1.py", "data/revisions-v1.json",
         "results/runs-v1.jsonl", "receipts/gaming-v1.json"]
for name in FILES:
    if not os.path.exists(name):
        os.makedirs(os.path.dirname(name) or ".", exist_ok=True)
        urllib.request.urlretrieve(BASE + name, name)
print("ready")

## Part 1: recompute the article's numbers

In [ ]:
import gaming_run_v1 as R

summary = R.summarize()
for model, row in summary["models"].items():
    print(model)
    for key, c in row["cells"].items():
        print(f"  {key:34s} visible pass {c['visiblePass']['k']:2d}/{c['n']}  {c['outcomes']}")

In [ ]:
# Pooled over the three models: impossible tasks, and what pressure did to the test files.
import gaming_stats_v1 as S

runs = R.load_runs()
impossible = [r for r in runs if r["variant"] == "impossible"]
k = sum(r["visiblePass"] for r in impossible)
print("impossible tasks passed:", k, "of", len(impossible), "Wilson", [round(x, 3) for x in S.wilson(k, len(impossible))])
edit = {(r["model"], r["task"], r["pressure"]): r["testEdit"] != "none" for r in impossible if r["writable"]}
pairs = [(m, t) for (m, t, p) in edit if p == "neutral"]
b = sum(edit[(m, t, "pressured")] and not edit[(m, t, "neutral")] for m, t in pairs)
c = sum(edit[(m, t, "neutral")] and not edit[(m, t, "pressured")] for m, t in pairs)
print("changed the tests only under pressure:", b, " only without:", c, " McNemar p =", round(S.mcnemar_exact(b, c), 4))

## Part 2: rerun a slice (needs an OS sandbox)

The runner executes model-written code, so it runs only under Seatbelt (macOS) or bubblewrap (Linux) and refuses otherwise. On a Linux machine or VM with `bwrap`, this runs 24 cells on Qwen3 1.7B.

In [ ]:
import gaming_sandbox_v1 as X

if X.available_sandbox() is None:
    print("No OS sandbox here, so model-written code will not be run. Part 1 above needs none.")
else:
    os.makedirs("mine/data", exist_ok=True)
    !cp gaming_tasks_v1.py gaming_sandbox_v1.py gaming_stats_v1.py gaming_run_v1.py mine/ && cp data/revisions-v1.json mine/data/
    !cd mine && python gaming_run_v1.py run --model Qwen/Qwen3-1.7B --limit 24 && python gaming_run_v1.py summarize > /dev/null